<a href="https://colab.research.google.com/github/ipreencekmr/anthropic/blob/main/Build_MCP_Tools_Resources_%26_Prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FastMCP — Build & Use an MCP Server in Colab

This notebook shows the smallest useful example of [FastMCP](https://gofastmcp.com):

1. **Build** an MCP server with a couple of tools using `@mcp.tool`.
2. **Connect** to it with an MCP `Client` — first in-memory (no network needed), then over real HTTP.
3. **Call** the tools like an AI agent would.

No API keys or external accounts needed — everything runs inside this Colab runtime.

## 1. Install FastMCP

In [1]:
!pip install -q fastmcp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.4/229.4 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 28.1 MB/s eta 0:00:00


## 2. Define the MCP server

An MCP server just wraps normal Python functions. FastMCP turns each `@mcp.tool`-decorated
function into a tool an AI client can discover and call, using the type hints and docstring
to build the schema automatically.

In [3]:
from fastmcp import FastMCP

mcp = FastMCP(name="Demo Server")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@mcp.tool
def reverse_text(text: str) -> str:
    """Reverse a string."""
    return text[::-1]

@mcp.tool
def word_count(text: str) -> dict:
    """Count words and characters in a piece of text."""
    return {"words": len(text.split()), "characters": len(text)}

print("Server ready:", mcp.name)


Server ready: Demo Server


## 3. Talk to it with an in-memory Client

For quick testing, `Client(mcp)` connects straight to the server object in the same
Python process — no subprocess, no port, no network. This is the fastest way to try
out a server while you're building it.

In [4]:
from fastmcp import Client

async def demo_in_memory():
    async with Client(mcp) as client:
        # Discover what the server exposes
        tools = await client.list_tools()
        print("Available tools:", [t.name for t in tools])

        # Call each tool
        r1 = await client.call_tool("add", {"a": 5, "b": 7})
        print("add(5, 7) ->", r1.data)

        r2 = await client.call_tool("reverse_text", {"text": "FastMCP"})
        print("reverse_text('FastMCP') ->", r2.data)

        r3 = await client.call_tool("word_count", {"text": "MCP servers are easy with FastMCP"})
        print("word_count(...) ->", r3.data)

# Colab supports top-level await in a code cell
await demo_in_memory()


Available tools: ['add', 'reverse_text', 'word_count']
add(5, 7) -> 12
reverse_text('FastMCP') -> PCMtsaF
word_count(...) -> {'words': 6, 'characters': 33}


## 4. (Bonus) Run it as a real HTTP server

The in-memory client is great for testing, but a real MCP server usually runs as its
own process and is reached over HTTP (this is how Claude Desktop, Claude Code, or any
other MCP-aware app would connect to it remotely).

Here we start the server on `localhost` inside the Colab VM using a background thread,
then connect to it with a client that speaks HTTP instead of talking to the Python
object directly.

In [5]:
import threading
import time

def run_server():
    # streamable-http is the standard transport for networked MCP servers
    mcp.run(transport="http", host="127.0.0.1", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # give the server a moment to start
print("Server running at http://127.0.0.1:8000/mcp")


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.2                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      Demo Server, 3.4.2                          │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[07/01/26 05:41:19] INFO     Starting MCP server 'Demo Server' with transport 'http' on            ]8;id=809288;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=728813;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#304\304]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [4978]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Server running at http://127.0.0.1:8000/mcp


In [6]:
from fastmcp import Client

async def demo_http():
    async with Client("http://127.0.0.1:8000/mcp") as client:
        await client.ping()
        print("Connected over HTTP!")

        result = await client.call_tool("add", {"a": 100, "b": 23})
        print("add(100, 23) ->", result.data)

await demo_http()


INFO:     127.0.0.1:35494 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35500 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:35502 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35504 - "POST /mcp HTTP/1.1" 200 OK
Connected over HTTP!
INFO:     127.0.0.1:35520 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:35530 - "POST /mcp HTTP/1.1" 200 OK
add(100, 23) -> 123
INFO:     127.0.0.1:35536 - "DELETE /mcp HTTP/1.1" 200 OK


## 5. Add Resources and Prompts

Tools aren't the only thing an MCP server can expose:

- **Resources** (`@mcp.resource`) are read-only data the client can fetch by URI — think of
  them like GET endpoints (files, config, records, generated text).
- **Prompts** (`@mcp.prompt`) are reusable, parameterized message templates the client can
  pull and hand to an LLM to kick off a task.

We add one of each to the same server.

In [7]:
# A simple in-memory "database" the resource will read from
NOTES = {
    "welcome": "Welcome to the FastMCP demo server!",
    "changelog": "v1.0 - added add, reverse_text, word_count tools.",
}

@mcp.resource("notes://{key}")
def get_note(key: str) -> str:
    """Fetch a note by key (e.g. 'welcome', 'changelog')."""
    return NOTES.get(key, f"No note found for '{key}'")

@mcp.prompt
def summarize_prompt(text: str) -> str:
    """Build a prompt asking an LLM to summarize the given text in one sentence."""
    return f"Summarize the following text in exactly one sentence:\n\n{text}"

print("Added resource 'notes://{key}' and prompt 'summarize_prompt'")

Added resource 'notes://{key}' and prompt 'summarize_prompt'


### Use the new resource and prompt from a client

Same `Client` API as before — just two new methods: `read_resource` and `get_prompt`.

In [8]:
async def demo_resources_and_prompts():
    async with Client(mcp) as client:
        # --- Resources ---
        resources = await client.list_resources()
        print("Available resource templates:", [r.uri for r in resources] or "(dynamic template, see below)")

        note = await client.read_resource("notes://welcome")
        print("notes://welcome ->", note[0].text)

        note2 = await client.read_resource("notes://changelog")
        print("notes://changelog ->", note2[0].text)

        # --- Prompts ---
        prompts = await client.list_prompts()
        print("Available prompts:", [p.name for p in prompts])

        rendered = await client.get_prompt("summarize_prompt", {"text": "FastMCP makes building MCP servers fast and simple."})
        print("summarize_prompt(...) ->", rendered.messages[0].content.text)

await demo_resources_and_prompts()


Available resource templates: (dynamic template, see below)
notes://welcome -> Welcome to the FastMCP demo server!
notes://changelog -> v1.0 - added add, reverse_text, word_count tools.
Available prompts: ['summarize_prompt']
summarize_prompt(...) -> Summarize the following text in exactly one sentence:

FastMCP makes building MCP servers fast and simple.


- Docs: [gofastmcp.com](https://gofastmcp.com)